# Gerador de trilhas Metroidvania 8-bit com gramáticas generativas

Pipeline: **gramática harmônica** → progressão de acordes → **gramática melódica** por acorde → canais no estilo NES (melodia, arpejo, baixo, bateria) → **WAV 8-bit** (e MIDI opcional).

Cada **subestilo** representa um tipo de área ou momento de um jogo Metroidvania:

| Subestilo | Inspiração | Clima |
|---|---|---|
| `castelo` | corredores góticos (Castlevania) | menor harmônica, arpejos e escalas rápidas |
| `caverna` | exploração atmosférica (Metroid) | acordes sus2, notas longas, pausas e eco |
| `chefe` | batalha contra chefe | rápido, nota pedal, `bII` e `bV` |
| `santuario` | sala de save / vila | maior, lento, sem bateria, com eco |
| `heroico` | tela de título / abertura | mixolídio, fanfarras com ritmo pontuado |
| `aquatico` | área submersa | dórico, frases fluidas, vibrato |

Para trocar, altere `SUBESTILO` na configuração e rode tudo (**Kernel → Restart & Run All**). O bloco 12 gera a trilha completa, com um WAV por subestilo.

## 1. Importações

In [ ]:
import random
import wave
import numpy as np
import mido
from IPython.display import Audio, display
from music21 import stream, meter, tempo, note, chord, instrument

## 2. Configuração
- `SUBESTILO`: um dos subestilos do bloco 7
- `TOM`: tônica (uma das notas de `NOTAS`). Subestilos menores ficam no tom menor dessa nota (`'A'` → Lá menor) e os maiores no tom maior
- `REPETICOES`: quantas vezes a progressão se repete. Trilhas de jogo vivem de *loop*, e a repetição faz os motivos voltarem
- `SEMENTE`: um número fixo repete o mesmo resultado; `None` varia a cada execução

In [ ]:
SUBESTILO = random.choice(['castelo','aquatico', 'heroico', 'santuario', 'chefe', 'caverna' ])
TOM = 'A'
REPETICOES = 2
SEMENTE = None

random.seed(SEMENTE)
np.random.seed(SEMENTE)

## 3. Teoria: notas, qualidades de acorde e graus
Cada grau é contado a partir da fundamental do acorde. Além de maior e menor, há acordes **diminutos** (`dim`, tensão gótica) e **sus2** (sem terça, som "suspenso" de caverna). Num acorde sus2, o grau `'3'` vira a segunda, para as gramáticas continuarem funcionando sem mudança.

In [ ]:
NOTAS = ['C','Db','D','Eb','E','F','Gb','G','Ab','A','Bb','B']

INTERVALOS = {
    'maior': {'1':0, '2':2, 'b3':3, '3':4, '4':5, '5':7, '6':9, 'b7':10},
    'menor': {'1':0, '2':2, '3':3, '4':5, '5':7, '6':8, 'b7':10},
    'dim':   {'1':0, '2':2, '3':3, '4':5, '5':6, '6':9, 'b7':9},
    'sus2':  {'1':0, '2':2, '3':2, '4':5, '5':7, '6':8, 'b7':10},
}

def parse_acorde(nome):
    """'Asus2' -> ('A','sus2'); 'Abdim' -> ('Ab','dim'); 'Am' -> ('A','menor'); 'C' -> ('C','maior')."""
    for sufixo, qualidade in (('sus2', 'sus2'), ('dim', 'dim'), ('m', 'menor')):   # 'dim' antes de 'm'!
        if nome.endswith(sufixo):
            return nome[:-len(sufixo)], qualidade
    return nome, 'maior'

def graus_do_acorde(nome):
    fund, qualidade = parse_acorde(nome)
    base = NOTAS.index(fund)
    return {g: NOTAS[(base + st) % 12] for g, st in INTERVALOS[qualidade].items()}

def m21(nome, oitava):
    """'Eb', 4 -> 'E-4' (notação do music21)."""
    return nome[0] + nome[1:].replace('b', '-') + str(oitava)

def nota(grau, acorde, oitava_base):
    """Grau com marcação de oitava (\"1'\" = uma oitava acima) -> nota do music21.
    Graus que não existem no acorde caem na fundamental."""
    oitava = oitava_base + grau.count("'")
    grau = grau.rstrip("'")
    graus = graus_do_acorde(acorde)
    return m21(graus.get(grau, graus['1']), oitava)

## 4. Motor da gramática
A mesma função expande qualquer gramática probabilística: cada símbolo tem uma lista de `(expansão, peso)`. Símbolos que não são chave do dicionário são terminais (graus, durações e `R` para pausa).

In [ ]:
def gerar(simbolo, regras):
    if simbolo not in regras:
        return [simbolo]
    opcoes, pesos = zip(*regras[simbolo])
    escolha = random.choices(opcoes, weights=pesos)[0]
    return [t for s in escolha.split() for t in gerar(s, regras)]

## 5. Gramáticas harmônicas
Os graus são relativos ao `TOM`. Minúsculas são acordes menores, `°` é diminuto e `sus2` é suspenso. Toda `PECA` termina num acorde que leva de volta ao começo, para o *loop* soar natural (o chefe termina em `V`, criando tensão antes de recomeçar).

In [ ]:
GRAUS_ACORDE = {
    # maiores e menores diatônicos
    'I': (0, ''), 'ii': (2, 'm'), 'iii': (4, 'm'), 'IV': (5, ''), 'V': (7, ''), 'vi': (9, 'm'),
    'i': (0, 'm'), 'iv': (5, 'm'), 'v': (7, 'm'),
    # empréstimos modais
    'bII': (1, ''), 'bIII': (3, ''), 'bV': (6, ''), 'bVI': (8, ''), 'bVII': (10, ''),
    # diminutos e suspensos
    'vii°': (11, 'dim'), 'ii°': (2, 'dim'),
    'isus2': (0, 'sus2'), 'bVIsus2': (8, 'sus2'), 'bVIIsus2': (10, 'sus2'),
}

def grau_para_acorde(grau, tom):
    st, sufixo = GRAUS_ACORDE[grau]
    return NOTAS[(NOTAS.index(tom) + st) % 12] + sufixo

HARMONIA = {
    'castelo': {
        'PECA':  [('FRASE FRASE FINAL', 1.0)],
        'FRASE': [('i bVI bVII i', 0.30), ('i iv V i', 0.30), ('i bVI iv V', 0.20), ('i vii° i V', 0.20)],
        'FINAL': [('bVI bVII i i', 0.5), ('iv V i i', 0.5)],
    },
    'caverna': {
        'PECA':  [('FRASE FRASE', 1.0)],
        'FRASE': [('isus2 isus2 bVIIsus2 isus2', 0.35), ('i bVI i bVI', 0.25),
                  ('isus2 bII isus2 i', 0.20), ('i iv i bVIIsus2', 0.20)],
    },
    'chefe': {
        'PECA':  [('FRASE FRASE FRASE FINAL', 1.0)],
        'FRASE': [('i bII i bII', 0.30), ('i bVI bV V', 0.25), ('i i bVI V', 0.25), ('i bIII bII i', 0.20)],
        'FINAL': [('bVI bV V V', 0.5), ('bII bII V V', 0.5)],
    },
    'santuario': {
        'PECA':  [('FRASE FINAL', 1.0)],
        'FRASE': [('I iii IV I', 0.30), ('I vi IV V', 0.30), ('I IV I V', 0.20), ('I bVII IV I', 0.20)],
        'FINAL': [('IV V I I', 0.6), ('IV iv I I', 0.4)],      # iv: cadência plagal menor, bem nostálgica
    },
    'heroico': {
        'PECA':  [('FRASE FRASE FINAL', 1.0)],
        'FRASE': [('I bVII IV I', 0.35), ('I V vi IV', 0.25), ('I bVI bVII I', 0.25), ('vi IV I V', 0.15)],
        'FINAL': [('bVI bVII I I', 0.6), ('IV V I I', 0.4)],
    },
    'aquatico': {
        'PECA':  [('FRASE FRASE', 1.0)],
        'FRASE': [('i IV i IV', 0.35), ('i ii i IV', 0.20), ('i bIII IV i', 0.25), ('i bVII IV i', 0.20)],
    },
}

## 6. Gramáticas melódicas
Cada compasso tem duas metades de 2 tempos. **FORTE** são notas de apoio (do acorde), **FRACA** são notas de passagem e `R` é pausa. Os blocos prontos (`ARPEJO`, `ESCALA`, `PEDAL`, `FANFARRA`, `FLUXO`) são os clichês melódicos de cada clima.

In [ ]:
MELODIA = {
    'castelo': {
        'MELODIA': [('METADE METADE', 1.0)],
        'METADE': [('ARPEJO', 0.25), ('ESCALA', 0.20),
                   ('FORTE 0.5 FRACA 0.25 FRACA 0.25 FORTE 1.0', 0.20),
                   ('FORTE 1.5 FRACA 0.5', 0.20), ('FORTE 0.75 FORTE 0.75 FRACA 0.5', 0.15)],
        'ARPEJO': [("1 0.25 3 0.25 5 0.25 1' 0.25 5 0.25 3 0.25 1 0.25 3 0.25", 0.5),
                   ("1 0.25 5 0.25 1' 0.25 5 0.25 3 0.25 5 0.25 1' 0.25 5 0.25", 0.5)],
        'ESCALA': [("1' 0.25 b7 0.25 6 0.25 5 0.25 4 0.25 3 0.25 2 0.25 1 0.25", 1.0)],
        'FORTE': [('1', 0.35), ('3', 0.30), ('5', 0.25), ("1'", 0.10)],
        'FRACA': [('2', 0.35), ('4', 0.20), ('5', 0.25), ('6', 0.20)],
    },
    'caverna': {
        'MELODIA': [('METADE METADE', 1.0)],
        'METADE': [('FORTE 2.0', 0.30), ('R 1.0 FORTE 1.0', 0.20), ('FORTE 1.5 FRACA 0.5', 0.20),
                   ('R 2.0', 0.15), ('FORTE 1.0 R 1.0', 0.15)],
        'FORTE': [('1', 0.30), ('5', 0.30), ("1'", 0.20), ('3', 0.20)],
        'FRACA': [('2', 0.40), ('6', 0.30), ('4', 0.30)],
    },
    'chefe': {
        'MELODIA': [('METADE METADE', 1.0)],
        'METADE': [('PEDAL', 0.30), ('FORTE 0.25 FORTE 0.25 FRACA 0.5 FORTE 0.5 FRACA 0.5', 0.25),
                   ('ARPEJO_DESC', 0.25), ('FORTE 0.5 FORTE 0.5 FORTE 1.0', 0.20)],
        'PEDAL': [("1 0.25 1' 0.25 1 0.25 b7 0.25 1 0.25 6 0.25 1 0.25 5 0.25", 1.0)],
        'ARPEJO_DESC': [("1' 0.25 5 0.25 3 0.25 1 0.25 5 0.25 3 0.25 1 0.25 R 0.25", 1.0)],
        'FORTE': [('1', 0.35), ("1'", 0.25), ('5', 0.25), ('3', 0.15)],
        'FRACA': [('2', 0.30), ('b7', 0.30), ('6', 0.20), ('4', 0.20)],
    },
    'santuario': {
        'MELODIA': [('METADE METADE', 1.0)],
        'METADE': [('FORTE 2.0', 0.30), ('FORTE 1.0 FRACA 1.0', 0.30),
                   ('FORTE 1.5 FRACA 0.5', 0.20), ('FORTE 1.0 R 1.0', 0.20)],
        'FORTE': [('3', 0.35), ('5', 0.30), ('1', 0.20), ("1'", 0.15)],
        'FRACA': [('2', 0.40), ('6', 0.40), ('5', 0.20)],
    },
    'heroico': {
        'MELODIA': [('METADE METADE', 1.0)],
        'METADE': [('FANFARRA', 0.30), ('FORTE 0.75 FRACA 0.25 FORTE 1.0', 0.30),
                   ('FORTE 1.0 FORTE 0.5 FRACA 0.5', 0.20), ('FORTE 2.0', 0.20)],
        'FANFARRA': [("1 0.5 5 0.5 1' 1.0", 0.5), ("5 0.75 5 0.25 1' 1.0", 0.5)],
        'FORTE': [('1', 0.30), ('5', 0.30), ('3', 0.25), ("1'", 0.15)],
        'FRACA': [('2', 0.40), ('6', 0.35), ('4', 0.25)],
    },
    'aquatico': {
        'MELODIA': [('METADE METADE', 1.0)],
        'METADE': [('FLUXO', 0.40), ('FORTE 1.0 FRACA 0.5 FRACA 0.5', 0.30), ('FORTE 1.5 FRACA 0.5', 0.30)],
        'FLUXO': [('1 0.5 2 0.5 3 0.5 5 0.5', 0.5), ('5 0.5 3 0.5 2 0.5 1 0.5', 0.5)],
        'FORTE': [('1', 0.30), ('3', 0.30), ('5', 0.30), ("1'", 0.10)],
        'FRACA': [('2', 0.35), ('4', 0.35), ('b7', 0.30)],
    },
}

## 7. Subestilos: andamento, arpejo, baixo, bateria e timbre
Aqui fica tudo que não é gramática, e é o lugar para criar subestilos novos:
- `arpejo`: `(ciclo de graus, duração de cada nota, oitava)`. O chip do NES não toca acordes, então a harmonia vira arpejo
- `baixo`: lista de `(grau, duração)` que soma 4 tempos, tocada no canal triângulo
- `bateria`: padrões de 16 semicolcheias, com `x` = golpe e `.` = silêncio, ou `None` para sem bateria
- `timbres`: forma de onda da melodia e do arpejo (`pulso12`, `pulso25`, `pulso50`, `triangulo`)
- `gate`: fração da duração em que a nota soa (menor = mais staccato)
- `eco`: `(atraso em tempos, volume)`, a técnica de NES de repetir a melodia atrasada para simular reverberação
- `vibrato`: profundidade da oscilação de afinação na melodia (0 = sem vibrato)
- `repetir`: chance de reaproveitar a melodia já usada para o mesmo acorde (cria motivos que voltam)

In [ ]:
SUBESTILOS = {
    'castelo': {
        'bpm': 150, 'arpejo': (['1', '3', '5', '3'], 0.25, 4),
        'baixo': [('1', 0.5), ("1'", 0.5)] * 4,
        'bateria': {'bumbo': 'x.......x.x.....', 'caixa': '....x.......x...', 'chimbal': 'x.x.x.x.x.x.x.x.'},
        'timbres': ('pulso25', 'pulso12'), 'gate': 0.85, 'eco': None, 'vibrato': 0.0, 'repetir': 0.6,
    },
    'caverna': {
        'bpm': 90, 'arpejo': (['1', '5', "1'", '5'], 0.5, 3),
        'baixo': [('1', 4.0)],
        'bateria': {'bumbo': 'x...............', 'caixa': '................', 'chimbal': '........x.......'},
        'timbres': ('pulso12', 'pulso12'), 'gate': 1.0, 'eco': (0.75, 0.4), 'vibrato': 0.004, 'repetir': 0.5,
    },
    'chefe': {
        'bpm': 180, 'arpejo': (['1', "1'", '5', "1'"], 0.25, 3),
        'baixo': [('1', 0.25), ('1', 0.25), ("1'", 0.25), ('1', 0.25)] * 4,
        'bateria': {'bumbo': 'x.x.x.x.x.x.x.x.', 'caixa': '....x.......x..x', 'chimbal': 'xxxxxxxxxxxxxxxx'},
        'timbres': ('pulso50', 'pulso25'), 'gate': 0.7, 'eco': None, 'vibrato': 0.0, 'repetir': 0.7,
    },
    'santuario': {
        'bpm': 80, 'arpejo': (['1', '3', '5', '3'], 1.0, 4),
        'baixo': [('1', 2.0), ('5', 2.0)],
        'bateria': None,
        'timbres': ('pulso50', 'pulso12'), 'gate': 1.0, 'eco': (0.5, 0.35), 'vibrato': 0.003, 'repetir': 0.6,
    },
    'heroico': {
        'bpm': 140, 'arpejo': (['1', '5', '3', '5'], 0.5, 4),
        'baixo': [('1', 0.5), ('1', 0.5), ('5', 0.5), ('5', 0.5)] * 2,
        'bateria': {'bumbo': 'x.....x.x.......', 'caixa': '....x.......x...', 'chimbal': 'x.x.x.x.x.x.x.x.'},
        'timbres': ('pulso50', 'pulso25'), 'gate': 0.85, 'eco': None, 'vibrato': 0.0, 'repetir': 0.6,
    },
    'aquatico': {
        'bpm': 100, 'arpejo': (['1', '5', "1'", "3'"], 0.5, 4),
        'baixo': [('1', 1.0), ('5', 1.0), ("1'", 1.0), ('5', 1.0)],
        'bateria': {'bumbo': 'x.......x.......', 'caixa': '................', 'chimbal': '..x...x...x...x.'},
        'timbres': ('pulso25', 'pulso12'), 'gate': 1.0, 'eco': (0.5, 0.3), 'vibrato': 0.006, 'repetir': 0.5,
    },
}

def validar_subestilo(nome, amostras=200):
    """Confere se melodia, baixo e bateria fecham exatamente um compasso de 4 tempos."""
    cfg = SUBESTILOS[nome]
    for _ in range(amostras):
        t = gerar('MELODIA', MELODIA[nome])
        total = sum(float(d) for d in t[1::2])
        assert len(t) % 2 == 0 and abs(total - 4) < 1e-9, f'{nome}: melodia soma {total} tempos: {t}'
    assert abs(sum(d for _, d in cfg['baixo']) - 4) < 1e-9, f'{nome}: baixo não soma 4 tempos'
    if cfg['bateria']:
        assert all(len(p) == 16 for p in cfg['bateria'].values()), f'{nome}: padrões de bateria precisam de 16 passos'
    assert nome in HARMONIA, f'{nome}: falta a gramática harmônica'

for nome in SUBESTILOS:
    validar_subestilo(nome)
print('Subestilos válidos:', ', '.join(SUBESTILOS))

## 8. Compositor
Gera a progressão, a melodia de cada acorde (reaproveitando motivos com a chance `repetir`), o arpejo, o baixo e a bateria. Devolve a partitura e a progressão.

In [ ]:
GOLPES = {'bumbo': 36, 'caixa': 38, 'chimbal': 42}

def nova_parte(nome):
    p = stream.Part()
    p.partName = nome
    inst = instrument.Instrument()
    inst.partName = nome
    p.insert(0, inst)
    return p

def compor(subestilo, tom='A', repeticoes=2):
    cfg = SUBESTILOS[subestilo]
    graus = gerar('PECA', HARMONIA[subestilo])
    progressao = [grau_para_acorde(g, tom) for g in graus] * repeticoes

    melodia, harmonia, baixo = nova_parte('Melodia'), nova_parte('Harmonia'), nova_parte('Baixo')
    melodia.insert(0, meter.TimeSignature('4/4'))
    melodia.insert(0, tempo.MetronomeMark(number=cfg['bpm']))
    partes = [melodia, harmonia, baixo]
    bateria = None
    if cfg['bateria']:
        bateria = nova_parte('Bateria')
        partes.append(bateria)

    memoria = {}
    for acorde in progressao:
        if acorde in memoria and random.random() < cfg['repetir']:
            terminais = memoria[acorde]
        else:
            terminais = gerar('MELODIA', MELODIA[subestilo])
            memoria[acorde] = terminais
        for grau, dur in zip(terminais[0::2], terminais[1::2]):
            dur = float(dur)
            if grau == 'R':
                melodia.append(note.Rest(quarterLength=dur))
            else:
                melodia.append(note.Note(nota(grau, acorde, 5), quarterLength=dur))

        # Harmonia em arpejo
        ciclo, dur_arp, oitava_arp = cfg['arpejo']
        for i in range(int(4 / dur_arp)):
            harmonia.append(note.Note(nota(ciclo[i % len(ciclo)], acorde, oitava_arp), quarterLength=dur_arp))

        # Baixo (canal triângulo)
        for grau, dur in cfg['baixo']:
            baixo.append(note.Note(nota(grau, acorde, 2), quarterLength=dur))

        # Bateria em 16 semicolcheias
        if bateria is not None:
            for passo in range(16):
                golpes = [GOLPES[k] for k, padrao in cfg['bateria'].items() if padrao[passo] == 'x']
                if golpes:
                    bateria.append(chord.Chord([note.Note(midi=g) for g in golpes], quarterLength=0.25))
                else:
                    bateria.append(note.Rest(quarterLength=0.25))

    return stream.Score(partes), progressao

partitura, progressao = compor(SUBESTILO, TOM, REPETICOES)
print(f'{SUBESTILO} em {TOM} ({SUBESTILOS[SUBESTILO]["bpm"]} BPM):', progressao)

## 9. Sintetizador 8-bit
Renderiza a partitura com as formas de onda do chip de som do NES: ondas de pulso (12,5%, 25%, 50%) na melodia e no arpejo, triângulo de 16 degraus no baixo e ruído na bateria. Eco e vibrato vêm da configuração do subestilo.

In [ ]:
SR = 44100
VOLUMES = {'Melodia': 0.30, 'Harmonia': 0.13, 'Baixo': 0.40}

def onda(tipo, freq, n, vibrato=0.0):
    t = np.arange(n) / SR
    f = freq * (1 + vibrato * np.sin(2 * np.pi * 5.5 * t))
    fase = np.cumsum(f) / SR % 1.0
    if tipo.startswith('pulso'):
        duty = {'pulso12': 0.125, 'pulso25': 0.25, 'pulso50': 0.5}[tipo]
        return np.where(fase < duty, 1.0, -1.0)
    if tipo == 'triangulo':
        return np.round((4 * np.abs(fase - 0.5) - 1) * 7.5) / 7.5    # 16 degraus, como no NES
    raise ValueError(tipo)

def envelope(n, ataque=0.002, soltura=0.01):
    env = np.ones(n)
    a, r = min(n, int(ataque * SR)), min(n, int(soltura * SR))
    if a: env[:a] = np.linspace(0, 1, a)
    if r: env[-r:] *= np.linspace(1, 0, r)
    return env

def tambor(midi_num):
    if midi_num == 36:                                    # bumbo: triângulo caindo de 150 para 50 Hz
        n = int(0.12 * SR)
        fase = np.cumsum(np.linspace(150, 50, n) / SR) % 1.0
        return 0.6 * (4 * np.abs(fase - 0.5) - 1) * np.linspace(1, 0, n)
    dur, vol = (0.12, 0.35) if midi_num == 38 else (0.03, 0.12)   # caixa / chimbal
    n = int(dur * SR)
    return vol * np.random.uniform(-1, 1, n) * np.exp(-np.linspace(0, 5, n))

def renderizar_8bit(partitura, subestilo, caminho):
    cfg = SUBESTILOS[subestilo]
    seg_tempo = 60 / cfg['bpm']
    timbre = {'Melodia': cfg['timbres'][0], 'Harmonia': cfg['timbres'][1], 'Baixo': 'triangulo'}
    mix = np.zeros(int(partitura.highestTime * seg_tempo * SR) + 2 * SR)

    for parte in partitura.parts:
        nome = parte.partName
        for el in parte.flatten().notes:
            ini = int(el.offset * seg_tempo * SR)
            if nome == 'Bateria':
                for p in el.pitches:
                    s = tambor(p.midi)
                    mix[ini:ini + len(s)] += s
                continue
            n = max(1, int(el.quarterLength * seg_tempo * SR * cfg['gate']))
            vib = cfg['vibrato'] if nome == 'Melodia' else 0.0
            for p in el.pitches:
                f = 440 * 2 ** ((p.midi - 69) / 12)
                som = VOLUMES[nome] * onda(timbre[nome], f, n, vib) * envelope(n)
                mix[ini:ini + n] += som
                if nome == 'Melodia' and cfg['eco']:
                    atraso, vol_eco = cfg['eco']
                    d = ini + int(atraso * seg_tempo * SR)
                    mix[d:d + n] += vol_eco * som

    mix = 0.9 * mix / max(1e-9, np.abs(mix).max())
    with wave.open(caminho, 'wb') as w:
        w.setnchannels(1)
        w.setsampwidth(2)
        w.setframerate(SR)
        w.writeframes((mix * 32767).astype(np.int16).tobytes())
    return caminho

## 10. Ouvir o subestilo escolhido

In [ ]:
arquivo_wav = renderizar_8bit(partitura, SUBESTILO, f'{SUBESTILO}_8bit.wav')
Audio(arquivo_wav)

## 11. (Opcional) Exportar MIDI
Útil para abrir numa DAW e trocar os timbres. O General MIDI não tem som de NES: a onda quadrada (programa 80) é a aproximação mais próxima, e o baixo usa Synth Bass (38). A bateria vai para o canal 10.

In [ ]:
PROGRAMAS_MIDI = {'Melodia': 80, 'Harmonia': 80, 'Baixo': 38}
CANAL_BATERIA = 9

def exportar_midi(partitura, caminho):
    partitura.write('midi', fp=caminho)
    nomes = [p.partName for p in partitura.parts]
    mid = mido.MidiFile(caminho)
    trilhas = [t for t in mid.tracks if any(m.type == 'note_on' for m in t)]
    for idx, (trilha, nome) in enumerate(zip(trilhas, nomes)):
        canal = CANAL_BATERIA if nome == 'Bateria' else idx
        programa = PROGRAMAS_MIDI.get(nome, 0)
        novas = [m for m in trilha if m.type != 'program_change']
        for m in novas:
            if hasattr(m, 'channel'):
                m.channel = canal
        pos = next((i for i, m in enumerate(novas) if not m.is_meta), len(novas))
        novas.insert(pos, mido.Message('program_change', channel=canal, program=programa, time=0))
        trilha[:] = novas
    mid.save(caminho)
    return caminho

print('MIDI salvo em', exportar_midi(partitura, f'{SUBESTILO}.mid'))

## 12. Trilha completa: um WAV por subestilo
Gera uma faixa para cada área do "jogo", todas no mesmo `TOM`, como numa trilha sonora de verdade.

In [ ]:
for nome in SUBESTILOS:
    p, prog = compor(nome, TOM, REPETICOES)
    caminho = renderizar_8bit(p, nome, f'trilha_{nome}.wav')
    print(f'{nome} ({SUBESTILOS[nome]["bpm"]} BPM):', ' '.join(prog[:len(prog) // REPETICOES]))
    display(Audio(caminho))

## 13. Como criar um subestilo novo
1. Adicione uma gramática em `HARMONIA` (bloco 5), com o símbolo inicial `PECA`
2. Adicione uma gramática em `MELODIA` (bloco 6), com o símbolo inicial `MELODIA`, em que cada `METADE` some 2 tempos
3. Adicione a configuração em `SUBESTILOS` (bloco 7)
4. Rode o bloco 7: o `validar_subestilo` avisa se algum ritmo não fecha o compasso

Ideias: `gelo` (lídio com `II` maior, pulso 12,5%, staccato), `ruinas` (frígio com `bII` constante, lento), `fuga` (chefe ainda mais rápido, 200 BPM, só pulso 50%).